In [1]:
import torch

import torch.nn.functional as F

words = open('names.txt', 'r').read(). splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

In [2]:
block_size = 3   # how many characters of context feed each prediction

X, Y = [], []
for w in words:
    context = [0] * block_size           # start fully padded with '.' (index 0)
    for ch in w + '.':                   # walk the word, then the end token
        ix = stoi[ch]
        X.append(context)                # input:  3 chars of context
        Y.append(ix)                     # target: the character that follows
        context = context[1:] + [ix]     # slide the window one step right

X = torch.tensor(X)
Y = torch.tensor(Y)
print(X.shape, Y.shape)

torch.Size([228146, 3]) torch.Size([228146])


In [3]:
for x, y in zip(X[:8], Y[:8]):
    print(''.join(itos[i.item()] for i in x), '->', itos[y.item()])

... -> e
..e -> m
.em -> m
emm -> a
mma -> .
... -> o
..o -> l
.ol -> i


In [4]:
for x, y in zip(X[:8], Y[:8]):

    context_chars = []

    for i in x:
        number = i.item()
        character = itos[number]
        context_chars.append(character)

    context_string = ''.join(context_chars)

    target_number = y.item()
    target_character = itos[target_number]

    print(context_string, '->', target_character)


    

... -> e
..e -> m
.em -> m
emm -> a
mma -> .
... -> o
..o -> l
.ol -> i


In [5]:
C = torch.randn((27,2))

emb = C[X]
print(emb.shape)

torch.Size([228146, 3, 2])


In [6]:
print(C[X][13, 2])   # build the full [N,3,2] tensor, then read example 13, slot 2
print(C[X[13, 2]])   # read the integer at X[13,2] first, then index C once

tensor([ 0.0849, -1.3081])
tensor([ 0.0849, -1.3081])


In [7]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [8]:
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
print(h.shape)

torch.Size([228146, 100])


In [9]:
print(emb[2])

tensor([[-1.4729, -0.3612],
        [-1.4883,  0.2807],
        [-1.1420,  1.3767]])


In [10]:
print(emb[2].view(6))

tensor([-1.4729, -0.3612, -1.4883,  0.2807, -1.1420,  1.3767])


In [11]:
W2 = torch.randn((100, 27))   # weights: 100 hidden -> 27 character scores
b2 = torch.randn(27)          # one bias per character

logits = h @ W2 + b2          # [N, 27]
print(logits.shape)

torch.Size([228146, 27])


In [12]:
print(logits[0])

tensor([-13.4968, -12.6781,  19.2526,  -9.4951,   8.4763, -19.5536,   0.0770,
          3.1379,   1.8308,  14.5736,  -2.0507,   2.4957,  -1.1062,  -1.3760,
         22.9327,   1.4790,  12.8266,   1.6725,  -0.2646,  -9.8259,  -0.8156,
          1.1024,   3.6273,   1.7519, -18.5369,  -2.9280,  -5.4894])


In [13]:
counts = logits.exp()

row_sums = counts.sum(1, keepdim = True)

prob = counts / row_sums

row_indices = torch.arange(Y.shape[0])

correct_character_probs = prob[row_indices, Y]

log_probs = correct_character_probs.log()

loss = -log_probs.mean()

print(loss.item())

18.514179229736328


In [14]:
print(counts.sum(1, keepdim=True).shape)   # torch.Size([N, 1]) — column axis kept
print(counts.sum(1).shape)                 # torch.Size([N])    — axis collapses
print(prob[0].sum())                       # tensor(1.0000) — row 0 sums to 1
print(prob[torch.arange(5), Y[:5]])        # the 5 probs on the correct chars, examples 0-4

torch.Size([228146, 1])
torch.Size([228146])
tensor(1.)
tensor([3.4473e-19, 1.6383e-13, 7.7517e-12, 2.6662e-11, 6.1085e-08])


In [15]:
loss = F.cross_entropy(logits, Y)
print(loss.item())

18.51418113708496


In [16]:
parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters))   # 3481 — total numbers the net will learn

for p in parameters:
    p.requires_grad = True

for step in range(100):
    # forward pass
    emb = C[X]                                   # [N, 3, 2]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)    # [N, 100]
    logits = h @ W2 + b2                          # [N, 27]
    loss = F.cross_entropy(logits, Y)

    # backward pass
    for p in parameters:
        p.grad = None                             # reset every gradient to empty
    loss.backward()                               # fill each p.grad

    # update
    for p in parameters:
        p.data += -0.1 * p.grad                   # gradient-descent step

    print(loss.item())

3481
18.51418113708496
17.058019638061523
15.772984504699707
14.68301773071289
13.82260799407959
13.152311325073242
12.563454627990723
12.024810791015625
11.531759262084961
11.079232215881348
10.662229537963867
10.27622127532959
9.917486190795898
9.583335876464844
9.271626472473145
8.980474472045898
8.708083152770996
8.452879905700684
8.213582992553711
7.98930025100708
7.779471397399902
7.583836555480957
7.40219783782959
7.234233856201172
7.079257488250732
6.936235427856445
6.803845405578613
6.68068790435791
6.5654191970825195
6.456897735595703
6.354219913482666
6.256706237792969
6.163848400115967
6.075263023376465
5.990660190582275
5.909813404083252
5.832540512084961
5.758677959442139
5.688073635101318
5.6205644607543945
5.555972576141357
5.494110107421875
5.4347758293151855
5.377768516540527
5.32289981842041
5.269993782043457
5.218900203704834
5.169488430023193
5.121645450592041
5.0752763748168945
5.030302047729492
4.986650466918945
4.944260597229004
4.903076171875
4.863048553466797


In [17]:
for step in range(10000):
    # build a minibatch
    ix = torch.randint(0, X.shape[0], (32,))

    # forward pass
    emb = C[X[ix]]                               # [32, 3, 2]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)    # [32, 100]
    logits = h @ W2 + b2                          # [32, 27]
    loss = F.cross_entropy(logits, Y[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad

    if step % 1000 == 0:
        print(step, loss.item())

0 3.766143560409546
1000 2.566045045852661
2000 2.13838791847229
3000 2.4502105712890625
4000 2.381769895553589
5000 2.921736240386963
6000 2.4862005710601807
7000 2.3154959678649902
8000 2.407658576965332
9000 2.2430477142333984


In [18]:
ix = torch.randint(0, X.shape[0], (32,))
print(ix.shape)      # torch.Size([32])    — 32 random row indices
print(X[ix].shape)   # torch.Size([32, 3]) — those 32 contexts
print(Y[ix].shape)   # torch.Size([32])    — their 32 targets, aligned by the same ix

torch.Size([32])
torch.Size([32, 3])
torch.Size([32])


In [19]:
with torch.no_grad():
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    print(F.cross_entropy(logits, Y).item())

2.45957088470459


In [ ]:
lre = torch.linspace(-3, 0, 1000)   # 1000 exponents, evenly spaced from -3 to 0
lrs = 10**lre                        # turn them into learning rates: 0.001 ... 1.0